# 08. 12-Feature vs 79-Feature Model Comparison

## Experimental Validation of the Proposed Machine Learning Pipeline

This notebook performs a controlled experimental comparison between two feature representations of the CIC-IDS2017 network intrusion detection dataset:

1. **79-Feature Representation** — the broader feature representation before the final feature-selection stage.
2. **12-Feature Representation** — the compact feature representation obtained through the preprocessing and feature-selection pipeline developed in the previous notebooks.

The purpose of this experiment is to determine whether the proposed preprocessing and feature-selection approach can substantially reduce the number of input features while maintaining competitive classification performance and reducing computational requirements.

### Research Question

> Can a carefully preprocessed and feature-selected 12-feature representation achieve comparable intrusion-detection performance to a 79-feature representation while reducing dimensionality and computational cost?

### Experimental Principle

The comparison is designed as a controlled experiment. Both feature representations will be evaluated using the same target classes, train/test partitioning strategy, model algorithm, evaluation metrics, and evaluation procedure wherever applicable.

The experiment focuses on both:

- **Predictive performance**
- **Computational efficiency**

### Main Comparison

| Aspect | 79 Features | 12 Features |
|---|---:|---:|
| Feature count | 79 | 12 |
| Feature selection | No final reduction | Yes |
| Model algorithm | Same | Same |
| Training data | Same basis | Same basis |
| Evaluation data | Same basis | Same basis |
| Accuracy | Measured | Measured |
| Precision | Measured | Measured |
| Recall | Measured | Measured |
| Weighted F1 | Measured | Measured |
| Macro F1 | Measured | Measured |
| Training time | Measured | Measured |
| Prediction time | Measured | Measured |
| Class-wise performance | Evaluated | Evaluated |
| Confusion matrix | Generated | Generated |

### Why Macro F1 Is Important

The CIC-IDS2017 classification problem contains substantially imbalanced classes. Therefore, accuracy alone may not adequately represent performance on minority attack classes.

For this reason, **Macro F1** is treated as an important comparison metric because it gives equal importance to each class.

Weighted F1 is also reported because it reflects the contribution of each class according to its support.

### Relationship to Previous Notebooks

This notebook does not replace the previous notebooks. Instead, it uses their outputs to perform an additional experimental validation:

**Notebook 01 → EDA**

**Notebook 02 → Preprocessing**

**Notebook 03 → Feature Selection**

**Notebook 04 → Baseline Model Building**

**Notebook 05 → Hyperparameter Tuning**

**Notebook 06 → Model Comparison**

**Notebook 07 → Final Model Evaluation**

**Notebook 08 → 12-Feature vs 79-Feature Experimental Comparison**

### Expected Contribution

The purpose of this experiment is not to assume that the 12-feature representation must outperform the 79-feature representation.

Instead, the results will be used to determine whether the proposed pipeline provides an effective trade-off between:

- Predictive performance
- Feature dimensionality
- Training cost
- Prediction cost
- Model complexity

All conclusions in this notebook will be based on the measured experimental results.

In [1]:
# 1. Import Required Libraries

import os
import time
import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("All required libraries imported successfully.")

All required libraries imported successfully.


## 2. Experimental Design

To make the comparison meaningful, the two feature representations will be evaluated under the same model and evaluation framework.

### Experimental Groups

#### Group A — 79 Features

The broader feature representation will be prepared from the CIC-IDS2017 dataset and evaluated using the selected classification algorithm.

#### Group B — 12 Features

The final 12 features obtained from Notebook 03 will be evaluated using the same classification algorithm.

### Controlled Variables

The following factors will remain consistent between the two experiments:

- Target variable
- Classification task
- Train/test methodology
- Model algorithm
- Random state
- Class weighting strategy
- Evaluation metrics
- Evaluation procedure

### Independent Variable

The primary independent variable is:

> **Number of input features**

The two experimental conditions are:

- 79 features
- 12 features

### Dependent Variables

The measured outcomes are:

- Accuracy
- Precision
- Recall
- Weighted F1
- Macro F1
- Training time
- Prediction time
- Class-wise F1 performance
- Confusion matrix

This design allows the effect of dimensionality reduction to be examined independently from the choice of classifier.

In [2]:
# 2. Define Project Paths

processed_dir = "../data/processed"
raw_data_path = "../data/CIC_IDS2017.csv"
models_dir = "../models"

os.makedirs(models_dir, exist_ok=True)

print("Raw dataset path:")
print(raw_data_path)

print("\nProcessed data directory:")
print(processed_dir)

print("\nModels directory:")
print(models_dir)

Raw dataset path:
../data/CIC_IDS2017.csv

Processed data directory:
../data/processed

Models directory:
../models


## 3. Load the 79-Feature Dataset

The raw CIC-IDS2017 dataset is loaded for the broader-feature experiment.

Before model training, the dataset will be inspected to determine:

- Number of rows
- Number of columns
- Target column
- Missing values
- Infinite values
- Data types
- Available feature columns

No assumptions are made about the exact number of columns until the dataset is verified programmatically.

In [3]:
# 3. Load Raw CIC-IDS2017 Dataset

df_raw = pd.read_csv(raw_data_path)

print("Raw dataset loaded successfully.")
print("Dataset shape:", df_raw.shape)

print("\nFirst 5 rows:")
display(df_raw.head())

print("\nColumn count:", len(df_raw.columns))

Raw dataset loaded successfully.
Dataset shape: (1233163, 79)

First 5 rows:


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN



Column count: 79


## 4. Inspect the Raw Dataset

The dataset structure is inspected before creating the experimental feature sets.

The inspection includes:

- Column names
- Target variable identification
- Missing-value count
- Infinite-value count
- Duplicate rows
- Numeric and non-numeric columns

This step ensures that the experimental comparison is based on the actual dataset rather than assumptions about its structure.

In [4]:
# 4. Dataset Inspection

print("Dataset Information:")
print(df_raw.info())

print("\nMissing values:")
missing_summary = df_raw.isnull().sum()
print("Total missing values:", missing_summary.sum())

print("\nDuplicate rows:")
print("Total duplicate rows:", df_raw.duplicated().sum())

print("\nData types:")
print(df_raw.dtypes.value_counts())

print("\nColumn names:")
for i, column in enumerate(df_raw.columns, start=1):
    print(f"{i:03d}. {column}")

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 1233163 entries, 0 to 1233162
Data columns (total 79 columns):
 #   Column                        Non-Null Count    Dtype  
---  ------                        --------------    -----  
 0    Destination Port             1233163 non-null  int64  
 1    Flow Duration                1233163 non-null  int64  
 2    Total Fwd Packets            1233163 non-null  int64  
 3    Total Backward Packets       1233163 non-null  int64  
 4   Total Length of Fwd Packets   1233163 non-null  int64  
 5    Total Length of Bwd Packets  1233163 non-null  int64  
 6    Fwd Packet Length Max        1233163 non-null  int64  
 7    Fwd Packet Length Min        1233163 non-null  int64  
 8    Fwd Packet Length Mean       1233163 non-null  float64
 9    Fwd Packet Length Std        1233163 non-null  float64
 10  Bwd Packet Length Max         1233163 non-null  int64  
 11   Bwd Packet Length Min        1233163 non-null  int64  
 12   Bwd Packet Length

## 5. Identify the Target Variable

The classification target must be separated from the input features.

The target variable represents the network traffic class used for intrusion detection.

The target column will be verified before creating the 79-feature representation.

In [6]:
# 5. Inspect and Identify the Target Column

print("All dataset columns:\n")

for i, column in enumerate(df_raw.columns, start=1):
    print(f"{i:03d}. {repr(column)}")

print("\n" + "=" * 70)
print("Last 10 columns:")
print("=" * 70)

for column in df_raw.columns[-10:]:
    print(repr(column))

All dataset columns:

001. ' Destination Port'
002. ' Flow Duration'
003. ' Total Fwd Packets'
004. ' Total Backward Packets'
005. 'Total Length of Fwd Packets'
006. ' Total Length of Bwd Packets'
007. ' Fwd Packet Length Max'
008. ' Fwd Packet Length Min'
009. ' Fwd Packet Length Mean'
010. ' Fwd Packet Length Std'
011. 'Bwd Packet Length Max'
012. ' Bwd Packet Length Min'
013. ' Bwd Packet Length Mean'
014. ' Bwd Packet Length Std'
015. 'Flow Bytes/s'
016. ' Flow Packets/s'
017. ' Flow IAT Mean'
018. ' Flow IAT Std'
019. ' Flow IAT Max'
020. ' Flow IAT Min'
021. 'Fwd IAT Total'
022. ' Fwd IAT Mean'
023. ' Fwd IAT Std'
024. ' Fwd IAT Max'
025. ' Fwd IAT Min'
026. 'Bwd IAT Total'
027. ' Bwd IAT Mean'
028. ' Bwd IAT Std'
029. ' Bwd IAT Max'
030. ' Bwd IAT Min'
031. 'Fwd PSH Flags'
032. ' Bwd PSH Flags'
033. ' Fwd URG Flags'
034. ' Bwd URG Flags'
035. ' Fwd Header Length'
036. ' Bwd Header Length'
037. 'Fwd Packets/s'
038. ' Bwd Packets/s'
039. ' Min Packet Length'
040. ' Max Packet Leng

## 6. Prepare the 79-Feature Representation

The 79-feature branch represents the broader feature space used for comparison.

Only valid numeric predictor columns will be considered for model input. The target variable is excluded from the feature matrix.

The feature count is verified programmatically rather than assumed.

The purpose of this stage is to establish the higher-dimensional representation against which the proposed 12-feature representation can be compared.

In [ ]:
# 6. Load and Inspect the Raw CIC-IDS2017 Dataset

import os
import numpy as np
import pandas as pd

raw_data_path = "../data/CIC_IDS2017.csv"

print("Checking dataset path...")
print(raw_data_path)

if not os.path.exists(raw_data_path):
    raise FileNotFoundError(
        f"Dataset not found at: {raw_data_path}\n"
        "Please verify that CIC_IDS2017.csv is inside ../data/"
    )

# Load dataset
df_raw = pd.read_csv(raw_data_path)

print("\nDataset loaded successfully.")
print("=" * 70)

print("Dataset shape:", df_raw.shape)
print("Number of rows:", df_raw.shape[0])
print("Number of columns:", df_raw.shape[1])

print("\n" + "=" * 70)
print("ALL COLUMN NAMES")
print("=" * 70)

for i, column in enumerate(df_raw.columns, start=1):
    print(f"{i:03d}. {repr(column)}")

print("\n" + "=" * 70)
print("LAST 10 COLUMNS")
print("=" * 70)

for column in df_raw.columns[-10:]:
    print(repr(column))

Checking dataset path...
../data/CIC_IDS2017.csv

Dataset loaded successfully.
Dataset shape: (1233163, 79)
Number of rows: 1233163
Number of columns: 79

ALL COLUMN NAMES
001. ' Destination Port'
002. ' Flow Duration'
003. ' Total Fwd Packets'
004. ' Total Backward Packets'
005. 'Total Length of Fwd Packets'
006. ' Total Length of Bwd Packets'
007. ' Fwd Packet Length Max'
008. ' Fwd Packet Length Min'
009. ' Fwd Packet Length Mean'
010. ' Fwd Packet Length Std'
011. 'Bwd Packet Length Max'
012. ' Bwd Packet Length Min'
013. ' Bwd Packet Length Mean'
014. ' Bwd Packet Length Std'
015. 'Flow Bytes/s'
016. ' Flow Packets/s'
017. ' Flow IAT Mean'
018. ' Flow IAT Std'
019. ' Flow IAT Max'
020. ' Flow IAT Min'
021. 'Fwd IAT Total'
022. ' Fwd IAT Mean'
023. ' Fwd IAT Std'
024. ' Fwd IAT Max'
025. ' Fwd IAT Min'
026. 'Bwd IAT Total'
027. ' Bwd IAT Mean'
028. ' Bwd IAT Std'
029. ' Bwd IAT Max'
030. ' Bwd IAT Min'
031. 'Fwd PSH Flags'
032. ' Bwd PSH Flags'
033. ' Fwd URG Flags'
034. ' Bwd URG 

## 7. Prepare the Original 78-Feature Matrix

The raw dataset contains 79 columns, consisting of 78 predictor features and one target column (`Label`).

Before creating the feature matrix, leading and trailing whitespace is removed from the column names.

The target variable is then separated from the 78 predictor features.

No feature selection is performed at this stage because this branch is intended to preserve the original feature representation for comparison with the final 12-feature representation.

In [2]:
# 7. Prepare the Original 78-Feature Matrix

# Clean column names
df_raw.columns = df_raw.columns.str.strip()

# Define target column
target_column = "Label"

# Verify target column exists
if target_column not in df_raw.columns:
    raise ValueError(
        f"Target column '{target_column}' was not found."
    )

# Separate original features and target
X_raw = df_raw.drop(columns=[target_column])
y_raw = df_raw[target_column].copy()

print("=" * 70)
print("ORIGINAL FEATURE MATRIX")
print("=" * 70)

print("\nTarget column:", target_column)
print("Total dataset columns:", df_raw.shape[1])
print("Original input features:", X_raw.shape[1])
print("Target columns:", 1)
print("Number of samples:", df_raw.shape[0])

print("\nFeature matrix shape:", X_raw.shape)
print("Target shape:", y_raw.shape)

print("\nFirst 10 feature names:")
for i, column in enumerate(X_raw.columns[:10], start=1):
    print(f"{i:02d}. {column}")

print("\nLast 5 feature names:")
for i, column in enumerate(X_raw.columns[-5:], start=X_raw.shape[1] - 4):
    print(f"{i:02d}. {column}")

ORIGINAL FEATURE MATRIX

Target column: Label
Total dataset columns: 79
Original input features: 78
Target columns: 1
Number of samples: 1233163

Feature matrix shape: (1233163, 78)
Target shape: (1233163,)

First 10 feature names:
01. Destination Port
02. Flow Duration
03. Total Fwd Packets
04. Total Backward Packets
05. Total Length of Fwd Packets
06. Total Length of Bwd Packets
07. Fwd Packet Length Max
08. Fwd Packet Length Min
09. Fwd Packet Length Mean
10. Fwd Packet Length Std

Last 5 feature names:
74. Active Min
75. Idle Mean
76. Idle Std
77. Idle Max
78. Idle Min


## 8. Check Data Types of the Original Features

The original 78-feature representation is inspected to verify the data types of all predictor variables.

Machine learning models require numerical input, so any non-numeric predictor columns must be identified before model training.

This step does not perform feature selection. It only verifies whether the original feature representation is suitable for numerical machine learning after the required preprocessing steps.

In [3]:
# 8. Check Data Types of the Original Features

print("=" * 70)
print("DATA TYPE CHECK — ORIGINAL 78 FEATURES")
print("=" * 70)

# Count data types
dtype_counts = X_raw.dtypes.value_counts()

print("\nData type distribution:")
print(dtype_counts)

# Identify non-numeric columns
non_numeric_columns = X_raw.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("\nNumber of numeric features:",
      X_raw.select_dtypes(include=[np.number]).shape[1])

print("Number of non-numeric features:",
      len(non_numeric_columns))

if non_numeric_columns:
    print("\nNon-numeric feature columns:")
    for column in non_numeric_columns:
        print("-", column)
else:
    print("\nAll 78 original features are numeric.")

DATA TYPE CHECK — ORIGINAL 78 FEATURES

Data type distribution:
int64      54
float64    24
Name: count, dtype: int64

Number of numeric features: 78
Number of non-numeric features: 0

All 78 original features are numeric.


## 9. Check Missing and Infinite Values

The original 78-feature representation is checked for missing (`NaN`) and infinite (`+∞` or `-∞`) values.

These values must be identified before model training because most machine learning algorithms cannot directly process them.

The counts are calculated across the complete original feature matrix without removing or modifying any observations at this stage.

In [4]:
# 9. Check Missing and Infinite Values

print("=" * 70)
print("MISSING AND INFINITE VALUE CHECK")
print("=" * 70)

# Count missing values
missing_values = X_raw.isna().sum().sum()

# Count positive and negative infinite values
infinite_values = np.isinf(X_raw).sum().sum()

# Count finite values
finite_values = np.isfinite(X_raw).sum().sum()

print(f"\nTotal feature values: {X_raw.size:,}")
print(f"Missing values (NaN): {missing_values:,}")
print(f"Infinite values (+/-inf): {infinite_values:,}")
print(f"Finite values: {finite_values:,}")

# Feature-wise missing values
missing_by_feature = X_raw.isna().sum()

# Feature-wise infinite values
infinite_by_feature = np.isinf(X_raw).sum()

print("\nFeatures containing missing values:",
      (missing_by_feature > 0).sum())

print("Features containing infinite values:",
      (infinite_by_feature > 0).sum())

if (missing_by_feature > 0).any():
    print("\nMissing values by feature:")
    print(missing_by_feature[missing_by_feature > 0])

if (infinite_by_feature > 0).any():
    print("\nInfinite values by feature:")
    print(infinite_by_feature[infinite_by_feature > 0])

if missing_values == 0 and infinite_values == 0:
    print("\nAll original feature values are finite and non-missing.")

MISSING AND INFINITE VALUE CHECK

Total feature values: 96,186,714
Missing values (NaN): 111
Infinite values (+/-inf): 1,817
Finite values: 96,184,786

Features containing missing values: 1
Features containing infinite values: 2

Missing values by feature:
Flow Bytes/s    111
dtype: int64

Infinite values by feature:
Flow Bytes/s      853
Flow Packets/s    964
dtype: int64


## 10. Handle Missing and Infinite Values

The original 78-feature representation contains a small number of missing and infinite values in rate-based features.

To make the feature matrix suitable for machine learning, infinite values are first converted to missing values.

The resulting missing values are then replaced using the median of each feature.

The median is calculated from the available values of each feature, allowing the original 78-feature representation to be retained without removing the affected observations or features.

No feature selection is performed at this stage.

In [5]:
# 10. Handle Missing and Infinite Values

print("=" * 70)
print("HANDLING MISSING AND INFINITE VALUES")
print("=" * 70)

# Create a copy so the original raw feature matrix remains unchanged
X_raw_clean = X_raw.copy()

# Convert infinite values to NaN
X_raw_clean.replace([np.inf, -np.inf], np.nan, inplace=True)

# Count missing values after converting infinities
missing_before_imputation = X_raw_clean.isna().sum().sum()

print("\nMissing values after converting infinities to NaN:",
      missing_before_imputation)

# Store median values for each feature
raw_feature_medians = X_raw_clean.median()

# Fill missing values using feature-wise medians
X_raw_clean = X_raw_clean.fillna(raw_feature_medians)

# Verify remaining missing and infinite values
remaining_missing = X_raw_clean.isna().sum().sum()
remaining_infinite = np.isinf(X_raw_clean).sum().sum()

print("\nAfter median imputation:")
print("Remaining missing values:", remaining_missing)
print("Remaining infinite values:", remaining_infinite)

print("\nCleaned feature matrix shape:", X_raw_clean.shape)

if remaining_missing == 0 and remaining_infinite == 0:
    print("\nAll 78 original features are now finite and non-missing.")

HANDLING MISSING AND INFINITE VALUES

Missing values after converting infinities to NaN: 1928

After median imputation:
Remaining missing values: 0
Remaining infinite values: 0

Cleaned feature matrix shape: (1233163, 78)

All 78 original features are now finite and non-missing.


## 11. Check Constant Features in the Original Representation

The cleaned 78-feature representation is checked for constant features.

A constant feature contains the same value for every observation and therefore provides no variation for distinguishing between classes.

The number of unique values in each feature is examined after missing and infinite values have been handled.

No features are removed at this stage because this notebook is intended to preserve the original feature representation for comparison.

In [6]:
# 11. Check Constant Features in the Original Representation

print("=" * 70)
print("CONSTANT FEATURE CHECK")
print("=" * 70)

# Count unique values for every feature
unique_counts = X_raw_clean.nunique()

# Identify constant features
constant_features = unique_counts[unique_counts <= 1]

print("\nTotal original features:", X_raw_clean.shape[1])
print("Constant features:", len(constant_features))

if len(constant_features) > 0:
    print("\nConstant features:")
    for feature, count in constant_features.items():
        print(f"- {feature}: {count} unique value")
else:
    print("\nNo constant features found.")

print("\nMinimum number of unique values among features:",
      unique_counts.min())

print("Maximum number of unique values among features:",
      unique_counts.max())

CONSTANT FEATURE CHECK

Total original features: 78
Constant features: 10

Constant features:
- Bwd PSH Flags: 1 unique value
- Fwd URG Flags: 1 unique value
- Bwd URG Flags: 1 unique value
- CWE Flag Count: 1 unique value
- Fwd Avg Bytes/Bulk: 1 unique value
- Fwd Avg Packets/Bulk: 1 unique value
- Fwd Avg Bulk Rate: 1 unique value
- Bwd Avg Bytes/Bulk: 1 unique value
- Bwd Avg Packets/Bulk: 1 unique value
- Bwd Avg Bulk Rate: 1 unique value

Minimum number of unique values among features: 1
Maximum number of unique values among features: 727089


## 12. Check Duplicate Observations

The original dataset is checked for completely duplicated observations.

Duplicate rows may represent repeated network-flow records and can affect the distribution of the dataset if they are present in large numbers.

The duplicate count is reported for transparency.

No duplicate rows are removed in this comparison branch at this stage because the objective is to preserve the original feature representation before model comparison.

In [7]:
# 12. Check Duplicate Observations

print("=" * 70)
print("DUPLICATE OBSERVATION CHECK")
print("=" * 70)

# Count completely duplicated rows
duplicate_rows = df_raw.duplicated().sum()

# Calculate duplicate percentage
duplicate_percentage = (
    duplicate_rows / len(df_raw)
) * 100

print("\nTotal observations:", len(df_raw))
print("Duplicate observations:", duplicate_rows)
print(f"Duplicate percentage: {duplicate_percentage:.6f}%")

if duplicate_rows == 0:
    print("\nNo completely duplicated observations found.")
else:
    print("\nDuplicate observations are present.")

DUPLICATE OBSERVATION CHECK

Total observations: 1233163
Duplicate observations: 114482
Duplicate percentage: 9.283606%

Duplicate observations are present.


## 13. Final Sanity Check of the Original 78-Feature Representation

Before model preparation, the cleaned original feature matrix is validated to confirm that:

- all 78 predictor features are present,
- all features are numeric,
- no missing values remain,
- no infinite values remain,
- and the number of observations is unchanged.

This confirms that the original feature representation is ready for the next stage of the comparison experiment.

In [8]:
# 13. Final Sanity Check of the Original 78-Feature Representation

print("=" * 70)
print("FINAL SANITY CHECK — ORIGINAL 78 FEATURES")
print("=" * 70)

# Basic dimensions
print("\nFeature matrix shape:", X_raw_clean.shape)
print("Target shape:", y_raw.shape)

# Numeric feature check
numeric_feature_count = X_raw_clean.select_dtypes(
    include=[np.number]
).shape[1]

# Missing and infinite checks
remaining_missing = X_raw_clean.isna().sum().sum()
remaining_infinite = np.isinf(X_raw_clean).sum().sum()

# Observation count
same_row_count = len(X_raw_clean) == len(y_raw)

print("\nNumber of numeric features:", numeric_feature_count)
print("Missing values:", remaining_missing)
print("Infinite values:", remaining_infinite)
print("Feature/target row count matches:", same_row_count)

# Final validation
validation_passed = (
    X_raw_clean.shape[1] == 78
    and numeric_feature_count == 78
    and remaining_missing == 0
    and remaining_infinite == 0
    and same_row_count
)

print("\nValidation passed:", validation_passed)

if validation_passed:
    print("\nThe original 78-feature representation is ready for model preparation.")
else:
    raise ValueError(
        "Original 78-feature representation failed validation."
    )

FINAL SANITY CHECK — ORIGINAL 78 FEATURES

Feature matrix shape: (1233163, 78)
Target shape: (1233163,)

Number of numeric features: 78
Missing values: 0
Infinite values: 0
Feature/target row count matches: True

Validation passed: True

The original 78-feature representation is ready for model preparation.


## 14. Inspect the Target Distribution

The target variable is inspected before creating the training and evaluation sets.

The class counts and percentages are calculated to confirm the class distribution used in the original 78-feature experiment.

Because the dataset is imbalanced, stratified splitting will be used in the next stage to preserve the class proportions across the training and evaluation sets.

In [9]:
# 14. Inspect the Target Distribution

print("=" * 70)
print("TARGET CLASS DISTRIBUTION")
print("=" * 70)

# Class counts
class_counts = y_raw.value_counts()

# Class percentages
class_percentages = (
    y_raw.value_counts(normalize=True)
    .mul(100)
    .round(4)
)

# Display results
target_distribution = pd.DataFrame({
    "Count": class_counts,
    "Percentage": class_percentages
})

print("\nTarget distribution:")
print(target_distribution)

print("\nNumber of target classes:", y_raw.nunique())
print("Total observations:", len(y_raw))

TARGET CLASS DISTRIBUTION

Target distribution:
           Count  Percentage
Label                       
BENIGN    944240     76.5706
PortScan  158930     12.8880
DDoS      128027     10.3820
Bot         1966      0.1594

Number of target classes: 4
Total observations: 1233163


## 15. Create a Stratified Training and Evaluation Split

The cleaned 78-feature representation is divided into training and evaluation sets using an 80:20 stratified split.

Stratification is used to preserve the relative distribution of the four target classes in both subsets, which is particularly important because the dataset contains a highly imbalanced class distribution.

A fixed random state of 42 is used to make the split reproducible.

In [10]:
# 15. Create a Stratified Training and Evaluation Split

from sklearn.model_selection import train_test_split

# Create an 80:20 stratified split
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw_clean,
    y_raw,
    test_size=0.20,
    stratify=y_raw,
    random_state=42
)

print("=" * 70)
print("TRAINING / EVALUATION SPLIT — 78 FEATURES")
print("=" * 70)

print("\nTraining feature matrix:", X_train_raw.shape)
print("Evaluation feature matrix:", X_test_raw.shape)

print("Training target shape:", y_train_raw.shape)
print("Evaluation target shape:", y_test_raw.shape)

print("\nTraining class percentages:")
print(
    y_train_raw.value_counts(normalize=True)
    .mul(100)
    .round(4)
)

print("\nEvaluation class percentages:")
print(
    y_test_raw.value_counts(normalize=True)
    .mul(100)
    .round(4)
)

TRAINING / EVALUATION SPLIT — 78 FEATURES

Training feature matrix: (986530, 78)
Evaluation feature matrix: (246633, 78)
Training target shape: (986530,)
Evaluation target shape: (246633,)

Training class percentages:
Label
BENIGN      76.5706
PortScan    12.8880
DDoS        10.3819
Bot          0.1594
Name: proportion, dtype: float64

Evaluation class percentages:
Label
BENIGN      76.5705
PortScan    12.8880
DDoS        10.3822
Bot          0.1593
Name: proportion, dtype: float64


## 16. Standardize the Original 78-Feature Representation

The training and evaluation feature sets are standardized using `StandardScaler`.

The scaler is fitted only on the training data and then applied to both the training and evaluation sets.

This prevents information from the evaluation set from influencing the transformation learned from the training data.

All 78 original features are retained during this step.

In [11]:
# 16. Standardize the Original 78-Feature Representation

from sklearn.preprocessing import StandardScaler

print("=" * 70)
print("STANDARDIZATION — ORIGINAL 78 FEATURES")
print("=" * 70)

# Create scaler
scaler_raw = StandardScaler()

# Fit only on training data
X_train_raw_scaled = scaler_raw.fit_transform(X_train_raw)

# Apply the same scaler to evaluation data
X_test_raw_scaled = scaler_raw.transform(X_test_raw)

print("\nTraining scaled shape:", X_train_raw_scaled.shape)
print("Evaluation scaled shape:", X_test_raw_scaled.shape)

# Verify training statistics
train_means = X_train_raw_scaled.mean(axis=0)
train_stds = X_train_raw_scaled.std(axis=0)

print("\nMaximum absolute training mean:",
      np.abs(train_means).max())

print("Minimum training standard deviation:",
      train_stds.min())

print("Maximum training standard deviation:",
      train_stds.max())

# Verify no invalid values were introduced
print("\nMissing values in scaled training data:",
      np.isnan(X_train_raw_scaled).sum())

print("Infinite values in scaled training data:",
      np.isinf(X_train_raw_scaled).sum())

print("\nMissing values in scaled evaluation data:",
      np.isnan(X_test_raw_scaled).sum())

print("Infinite values in scaled evaluation data:",
      np.isinf(X_test_raw_scaled).sum())

STANDARDIZATION — ORIGINAL 78 FEATURES

Training scaled shape: (986530, 78)
Evaluation scaled shape: (246633, 78)

Maximum absolute training mean: 7.467494231661195e-17
Minimum training standard deviation: 0.0
Maximum training standard deviation: 1.0000000000000004

Missing values in scaled training data: 0
Infinite values in scaled training data: 0

Missing values in scaled evaluation data: 0
Infinite values in scaled evaluation data: 0


## 17. Load the Final 12-Feature Representation

The final 12-feature representation produced by the feature-selection pipeline is loaded from the processed datasets.

These features represent the reduced feature space used by the final Decision Tree model.

The previously generated training and evaluation partitions are used so that the 12-feature representation can be compared with the original 78-feature representation.

In [12]:
# 17. Load the Final 12-Feature Representation

selected_train_path = "../data/processed/X_train_selected.csv"
selected_test_path = "../data/processed/X_test_selected.csv"

print("=" * 70)
print("LOADING FINAL 12-FEATURE REPRESENTATION")
print("=" * 70)

# Load selected feature datasets
X_train_selected = pd.read_csv(selected_train_path)
X_test_selected = pd.read_csv(selected_test_path)

# Load corresponding targets
y_train_selected = pd.read_csv(
    "../data/processed/y_train_selected.csv"
).squeeze("columns")

y_test_selected = pd.read_csv(
    "../data/processed/y_test_selected.csv"
).squeeze("columns")

print("\nTraining selected-feature matrix:",
      X_train_selected.shape)

print("Evaluation selected-feature matrix:",
      X_test_selected.shape)

print("Training target shape:",
      y_train_selected.shape)

print("Evaluation target shape:",
      y_test_selected.shape)

print("\nSelected features:")
for i, feature in enumerate(X_train_selected.columns, start=1):
    print(f"{i:02d}. {feature}")

LOADING FINAL 12-FEATURE REPRESENTATION

Training selected-feature matrix: (894944, 12)
Evaluation selected-feature matrix: (223737, 12)
Training target shape: (894944,)
Evaluation target shape: (223737,)

Selected features:
01. Destination Port
02. Flow Duration
03. Total Fwd Packets
04. Total Length of Fwd Packets
05. Total Length of Bwd Packets
06. Fwd Packet Length Mean
07. Bwd Packet Length Min
08. Bwd Packets/s
09. PSH Flag Count
10. Init_Win_bytes_forward
11. Init_Win_bytes_backward
12. min_seg_size_forward


## 18. Compare the Sample Counts of Both Representations

The original 78-feature representation and the final 12-feature representation are compared in terms of their available training and evaluation observations.

The 78-feature branch contains the complete cleaned dataset, while the 12-feature datasets were generated by the earlier preprocessing pipeline.

Because the two representations currently contain different numbers of observations, their sample counts are recorded explicitly before model comparison.

This check prevents the comparison from incorrectly assuming that both representations contain identical observations.

In [13]:
# 18. Compare the Sample Counts of Both Representations

print("=" * 70)
print("SAMPLE COUNT COMPARISON")
print("=" * 70)

comparison_counts = pd.DataFrame({
    "Representation": [
        "Original 78 Features",
        "Final 12 Features"
    ],
    "Training Samples": [
        X_train_raw_scaled.shape[0],
        X_train_selected.shape[0]
    ],
    "Evaluation Samples": [
        X_test_raw_scaled.shape[0],
        X_test_selected.shape[0]
    ],
    "Features": [
        X_train_raw_scaled.shape[1],
        X_train_selected.shape[1]
    ]
})

print("\n", comparison_counts.to_string(index=False))

# Calculate differences
train_difference = (
    X_train_raw_scaled.shape[0] -
    X_train_selected.shape[0]
)

test_difference = (
    X_test_raw_scaled.shape[0] -
    X_test_selected.shape[0]
)

print("\nTraining sample difference:", train_difference)
print("Evaluation sample difference:", test_difference)

if train_difference == 0 and test_difference == 0:
    print("\nBoth representations contain the same number of observations.")
else:
    print("\nThe two representations contain different numbers of observations.")
    print("This difference will be considered when interpreting the comparison.")

SAMPLE COUNT COMPARISON

       Representation  Training Samples  Evaluation Samples  Features
Original 78 Features            986530              246633        78
   Final 12 Features            894944              223737        12

Training sample difference: 91586
Evaluation sample difference: 22896

The two representations contain different numbers of observations.
This difference will be considered when interpreting the comparison.


## 19. Align the 78-Feature Representation With the Final Dataset

The final 12-feature representation was produced after duplicate observations were removed during the preprocessing pipeline.

The original 78-feature branch is therefore aligned with the same duplicate-free dataset before the final model comparison.

This ensures that both feature representations are based on the same number of observations and allows a more controlled comparison of the effect of feature reduction.

No feature selection is performed on the 78-feature branch.

In [14]:
# 19. Align the 78-Feature Representation With the Final Dataset

print("=" * 70)
print("ALIGNING THE 78-FEATURE REPRESENTATION")
print("=" * 70)

# Remove duplicate observations from the cleaned original dataset
df_comparison = df_raw.copy()

initial_rows = len(df_comparison)

df_comparison = df_comparison.drop_duplicates().reset_index(drop=True)

remaining_rows = len(df_comparison)
removed_rows = initial_rows - remaining_rows

print("\nInitial observations:", initial_rows)
print("Duplicate observations removed:", removed_rows)
print("Remaining observations:", remaining_rows)

# Prepare aligned features and target
X_comparison_78 = df_comparison.drop(columns=["Label"])
y_comparison = df_comparison["Label"].copy()

print("\nAligned 78-feature matrix:", X_comparison_78.shape)
print("Aligned target shape:", y_comparison.shape)

# Verify against the final 12-feature dataset size
expected_rows = (
    X_train_selected.shape[0] +
    X_test_selected.shape[0]
)

print("\nExpected observations from 12-feature datasets:",
      expected_rows)

print("Observation count matches:",
      len(X_comparison_78) == expected_rows)

if len(X_comparison_78) != expected_rows:
    raise ValueError(
        "The aligned 78-feature dataset does not contain "
        "the same number of observations as the 12-feature dataset."
    )

print("\nThe 78-feature representation is now aligned "
      "with the duplicate-free dataset used by the 12-feature pipeline.")

ALIGNING THE 78-FEATURE REPRESENTATION

Initial observations: 1233163
Duplicate observations removed: 114482
Remaining observations: 1118681

Aligned 78-feature matrix: (1118681, 78)
Aligned target shape: (1118681,)

Expected observations from 12-feature datasets: 1118681
Observation count matches: True

The 78-feature representation is now aligned with the duplicate-free dataset used by the 12-feature pipeline.


## 20. Create the Aligned Training and Evaluation Split

The duplicate-free 78-feature representation is divided into training and evaluation sets using the same 80:20 stratified strategy.

The split is performed after duplicate removal so that the 78-feature representation contains the same total number of observations as the final 12-feature representation.

Stratification preserves the class distribution across both subsets, and a fixed random state of 42 ensures reproducibility.

In [15]:
# 20. Create the Aligned Training and Evaluation Split

from sklearn.model_selection import train_test_split

# Create aligned 80:20 stratified split
X_train_78, X_test_78, y_train_78, y_test_78 = train_test_split(
    X_comparison_78,
    y_comparison,
    test_size=0.20,
    stratify=y_comparison,
    random_state=42
)

print("=" * 70)
print("ALIGNED TRAINING / EVALUATION SPLIT — 78 FEATURES")
print("=" * 70)

print("\nTraining feature matrix:", X_train_78.shape)
print("Evaluation feature matrix:", X_test_78.shape)

print("Training target shape:", y_train_78.shape)
print("Evaluation target shape:", y_test_78.shape)

print("\nTraining class percentages:")
print(
    y_train_78.value_counts(normalize=True)
    .mul(100)
    .round(4)
)

print("\nEvaluation class percentages:")
print(
    y_test_78.value_counts(normalize=True)
    .mul(100)
    .round(4)
)

print("\nTotal observations:",
      len(X_train_78) + len(X_test_78))

ALIGNED TRAINING / EVALUATION SPLIT — 78 FEATURES

Training feature matrix: (894944, 78)
Evaluation feature matrix: (223737, 78)
Training target shape: (894944,)
Evaluation target shape: (223737,)

Training class percentages:
Label
BENIGN      80.2636
DDoS        11.4435
PortScan     8.1184
Bot          0.1745
Name: proportion, dtype: float64

Evaluation class percentages:
Label
BENIGN      80.2634
DDoS        11.4433
PortScan     8.1185
Bot          0.1748
Name: proportion, dtype: float64

Total observations: 1118681


## 21. Verify Target Alignment Between the Two Representations

The training and evaluation target distributions of the 78-feature representation are compared with the saved 12-feature representation.

Matching class counts provide evidence that both representations use the same stratified partition of the duplicate-free dataset. This check focuses on the target distribution before model training.

In [18]:
# 21. Verify target alignment after normalizing class labels

# Map the original text labels to the encoded labels
label_mapping = {
    "BENIGN": 0,
    "Bot": 1,
    "DDoS": 2,
    "PortScan": 3
}

# Convert 78-feature labels to the same encoding used by the 12-feature data
y_train_78_encoded = y_train_78.map(label_mapping)
y_test_78_encoded = y_test_78.map(label_mapping)

# Check that all labels were successfully mapped
if y_train_78_encoded.isna().any() or y_test_78_encoded.isna().any():
    raise ValueError("Some 78-feature labels could not be mapped.")

# Count classes
train_counts_78 = y_train_78_encoded.value_counts().sort_index()
test_counts_78 = y_test_78_encoded.value_counts().sort_index()

train_counts_12 = y_train_selected.value_counts().sort_index()
test_counts_12 = y_test_selected.value_counts().sort_index()

# Create comparison table
alignment_check = pd.DataFrame({
    "78_Feature_Train": train_counts_78,
    "12_Feature_Train": train_counts_12,
    "78_Feature_Evaluation": test_counts_78,
    "12_Feature_Evaluation": test_counts_12
}).fillna(0).astype(int)

print("=" * 70)
print("TARGET ALIGNMENT CHECK AFTER LABEL NORMALIZATION")
print("=" * 70)

print("\nClass-wise target counts:")
print(alignment_check)

# Compare the normalized class counts
train_alignment = train_counts_78.equals(train_counts_12)
test_alignment = test_counts_78.equals(test_counts_12)

print("\nTraining target counts match:", train_alignment)
print("Evaluation target counts match:", test_alignment)

if train_alignment and test_alignment:
    print("\nOverall target alignment: TRUE")
    print("The two representations have matching class counts.")
else:
    print("\nOverall target alignment: FALSE")

TARGET ALIGNMENT CHECK AFTER LABEL NORMALIZATION

Class-wise target counts:
   78_Feature_Train  12_Feature_Train  78_Feature_Evaluation  \
0            718314            718314                 179579   
1              1562              1562                    391   
2            102413            102413                  25603   
3             72655             72655                  18164   

   12_Feature_Evaluation  
0                 179579  
1                    391  
2                  25603  
3                  18164  

Training target counts match: True
Evaluation target counts match: True

Overall target alignment: TRUE
The two representations have matching class counts.


## 22. Handle Missing and Infinite Values Using Training-Only Statistics

The aligned 78-feature training and evaluation sets are checked for missing and infinite values.

Infinite values are first converted to missing values. Missing values are then imputed using feature-wise medians calculated **only from the training set**.

The same training-derived medians are applied to the evaluation set to prevent data leakage.

In [19]:
# 22. Handle missing and infinite values using training-only statistics

# Create copies to preserve the original aligned split
X_train_78_clean = X_train_78.copy()
X_test_78_clean = X_test_78.copy()

# Convert infinite values to NaN
X_train_78_clean.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test_78_clean.replace([np.inf, -np.inf], np.nan, inplace=True)

# Count missing values before imputation
train_missing_before = X_train_78_clean.isna().sum().sum()
test_missing_before = X_test_78_clean.isna().sum().sum()

# Calculate feature-wise medians using TRAINING DATA ONLY
train_feature_medians_78 = X_train_78_clean.median()

# Apply the training medians to both datasets
X_train_78_clean = X_train_78_clean.fillna(train_feature_medians_78)
X_test_78_clean = X_test_78_clean.fillna(train_feature_medians_78)

# Final validation
train_missing_after = X_train_78_clean.isna().sum().sum()
test_missing_after = X_test_78_clean.isna().sum().sum()

train_infinite_after = np.isinf(X_train_78_clean.to_numpy()).sum()
test_infinite_after = np.isinf(X_test_78_clean.to_numpy()).sum()

print("=" * 70)
print("MISSING / INFINITE VALUE HANDLING — 78 FEATURES")
print("=" * 70)

print("\nMissing values before imputation:")
print("Training:", train_missing_before)
print("Evaluation:", test_missing_before)

print("\nMissing values after imputation:")
print("Training:", train_missing_after)
print("Evaluation:", test_missing_after)

print("\nInfinite values after imputation:")
print("Training:", train_infinite_after)
print("Evaluation:", test_infinite_after)

print("\nTraining shape:", X_train_78_clean.shape)
print("Evaluation shape:", X_test_78_clean.shape)

validation_78 = (
    train_missing_after == 0
    and test_missing_after == 0
    and train_infinite_after == 0
    and test_infinite_after == 0
)

print("\nValidation:", validation_78)

MISSING / INFINITE VALUE HANDLING — 78 FEATURES

Missing values before imputation:
Training: 1262
Evaluation: 306

Missing values after imputation:
Training: 0
Evaluation: 0

Infinite values after imputation:
Training: 0
Evaluation: 0

Training shape: (894944, 78)
Evaluation shape: (223737, 78)

Validation: True


## 23. Define the Common Decision Tree Model

A Decision Tree classifier is used for both feature representations so that the comparison focuses on the effect of feature representation rather than differences in model configuration.

The same class-balancing strategy and random state used for the final Decision Tree model are retained.

In [20]:
# 23. Define the common Decision Tree model

from sklearn.tree import DecisionTreeClassifier

# Same model configuration used in the final model evaluation
dt_78 = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

dt_12 = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

print("=" * 70)
print("COMMON DECISION TREE CONFIGURATION")
print("=" * 70)

print("\n78-feature model:")
print(dt_78)

print("\n12-feature model:")
print(dt_12)

print("\nModel configurations identical:",
      dt_78.get_params() == dt_12.get_params())

COMMON DECISION TREE CONFIGURATION

78-feature model:
DecisionTreeClassifier(class_weight='balanced', random_state=42)

12-feature model:
DecisionTreeClassifier(class_weight='balanced', random_state=42)

Model configurations identical: True


## 24. Train the Decision Tree Using the 78-Feature Representation

The common Decision Tree classifier is trained using the original 78-feature representation.

Training time is recorded to measure the computational cost of using the broader feature representation.

In [21]:
# 24. Train the Decision Tree using the 78-feature representation

import time

print("=" * 70)
print("TRAINING DECISION TREE — 78 FEATURES")
print("=" * 70)

# Start training timer
start_time_78 = time.perf_counter()

# Train the model
dt_78.fit(X_train_78_clean, y_train_78_encoded)

# End training timer
training_time_78 = time.perf_counter() - start_time_78

print("\nTraining completed successfully.")
print("Training samples:", X_train_78_clean.shape[0])
print("Number of features:", X_train_78_clean.shape[1])
print(f"Training time: {training_time_78:.4f} seconds")

print("\nModel depth:", dt_78.get_depth())
print("Number of leaves:", dt_78.get_n_leaves())

TRAINING DECISION TREE — 78 FEATURES

Training completed successfully.
Training samples: 894944
Number of features: 78
Training time: 56.3488 seconds

Model depth: 51
Number of leaves: 644


## 25. Train the Decision Tree Using the 12-Feature Representation

The same Decision Tree configuration is trained using the final 12 selected features.

Training time is recorded using the same timing method as the 78-feature representation, allowing the computational cost of both feature representations to be compared fairly.

In [22]:
# 25. Train the Decision Tree using the 12-feature representation

print("=" * 70)
print("TRAINING DECISION TREE — 12 FEATURES")
print("=" * 70)

# Start training timer
start_time_12 = time.perf_counter()

# Train the model
dt_12.fit(X_train_selected, y_train_selected)

# End training timer
training_time_12 = time.perf_counter() - start_time_12

print("\nTraining completed successfully.")
print("Training samples:", X_train_selected.shape[0])
print("Number of features:", X_train_selected.shape[1])
print(f"Training time: {training_time_12:.4f} seconds")

print("\nModel depth:", dt_12.get_depth())
print("Number of leaves:", dt_12.get_n_leaves())

TRAINING DECISION TREE — 12 FEATURES

Training completed successfully.
Training samples: 894944
Number of features: 12
Training time: 7.5103 seconds

Model depth: 36
Number of leaves: 918


## 26. Measure Prediction Time for Both Representations

The trained Decision Tree models are used to generate predictions on their respective evaluation datasets.

Prediction time is measured separately for the 78-feature and 12-feature representations using the same timing method. This provides a direct comparison of inference cost.

In [23]:
# 26. Measure prediction time for both representations

print("=" * 70)
print("PREDICTION TIME COMPARISON")
print("=" * 70)

# Prediction using 78-feature representation
start_time_78_prediction = time.perf_counter()

y_pred_78 = dt_78.predict(X_test_78_clean)

prediction_time_78 = time.perf_counter() - start_time_78_prediction

# Prediction using 12-feature representation
start_time_12_prediction = time.perf_counter()

y_pred_12 = dt_12.predict(X_test_selected)

prediction_time_12 = time.perf_counter() - start_time_12_prediction

print("\n78-feature representation:")
print("Evaluation samples:", X_test_78_clean.shape[0])
print(f"Prediction time: {prediction_time_78:.4f} seconds")

print("\n12-feature representation:")
print("Evaluation samples:", X_test_selected.shape[0])
print(f"Prediction time: {prediction_time_12:.4f} seconds")

print("\nPredictions generated successfully:")
print("78-feature predictions:", len(y_pred_78))
print("12-feature predictions:", len(y_pred_12))

PREDICTION TIME COMPARISON

78-feature representation:
Evaluation samples: 223737
Prediction time: 0.2716 seconds

12-feature representation:
Evaluation samples: 223737
Prediction time: 0.0791 seconds

Predictions generated successfully:
78-feature predictions: 223737
12-feature predictions: 223737


## 27. Compare Classification Performance

The predictions from both Decision Tree models are evaluated using accuracy, weighted precision, weighted recall, weighted F1-score, and macro F1-score.

Macro F1 is given particular importance because the dataset is highly imbalanced and gives equal weight to each class regardless of its support.

In [24]:
# 27. Compare classification performance

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Calculate metrics for 78-feature representation
accuracy_78 = accuracy_score(y_test_78_encoded, y_pred_78)
weighted_precision_78 = precision_score(
    y_test_78_encoded,
    y_pred_78,
    average="weighted",
    zero_division=0
)
weighted_recall_78 = recall_score(
    y_test_78_encoded,
    y_pred_78,
    average="weighted",
    zero_division=0
)
weighted_f1_78 = f1_score(
    y_test_78_encoded,
    y_pred_78,
    average="weighted",
    zero_division=0
)
macro_f1_78 = f1_score(
    y_test_78_encoded,
    y_pred_78,
    average="macro",
    zero_division=0
)

# Calculate metrics for 12-feature representation
accuracy_12 = accuracy_score(y_test_selected, y_pred_12)
weighted_precision_12 = precision_score(
    y_test_selected,
    y_pred_12,
    average="weighted",
    zero_division=0
)
weighted_recall_12 = recall_score(
    y_test_selected,
    y_pred_12,
    average="weighted",
    zero_division=0
)
weighted_f1_12 = f1_score(
    y_test_selected,
    y_pred_12,
    average="weighted",
    zero_division=0
)
macro_f1_12 = f1_score(
    y_test_selected,
    y_pred_12,
    average="macro",
    zero_division=0
)

# Create comparison table
performance_comparison = pd.DataFrame({
    "Representation": [
        "78 Original Features",
        "12 Selected Features"
    ],
    "Accuracy": [
        accuracy_78,
        accuracy_12
    ],
    "Weighted Precision": [
        weighted_precision_78,
        weighted_precision_12
    ],
    "Weighted Recall": [
        weighted_recall_78,
        weighted_recall_12
    ],
    "Weighted F1": [
        weighted_f1_78,
        weighted_f1_12
    ],
    "Macro F1": [
        macro_f1_78,
        macro_f1_12
    ]
})

print("=" * 70)
print("CLASSIFICATION PERFORMANCE COMPARISON")
print("=" * 70)

print(
    performance_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

CLASSIFICATION PERFORMANCE COMPARISON
      Representation  Accuracy  Weighted Precision  Weighted Recall  Weighted F1  Macro F1
78 Original Features  0.999321            0.999304         0.999321     0.999311  0.955221
12 Selected Features  0.999169            0.999175         0.999169     0.999172  0.947471


## 28. Compare Class-Wise Performance

Class-wise precision, recall, and F1-score are calculated for both feature representations.

This analysis provides a more detailed view of model performance across individual traffic classes, particularly the minority Bot class, which may be hidden by the very high overall accuracy of the dataset.

In [25]:
# 28. Compare class-wise performance

from sklearn.metrics import classification_report

# Generate classification reports
report_78 = classification_report(
    y_test_78_encoded,
    y_pred_78,
    output_dict=True,
    zero_division=0
)

report_12 = classification_report(
    y_test_selected,
    y_pred_12,
    output_dict=True,
    zero_division=0
)

# Class labels
class_labels = ["0", "1", "2", "3"]

# Create class-wise comparison table
classwise_comparison = pd.DataFrame({
    "Class": class_labels,
    "78_Feature_Precision": [
        report_78[label]["precision"] for label in class_labels
    ],
    "12_Feature_Precision": [
        report_12[label]["precision"] for label in class_labels
    ],
    "78_Feature_Recall": [
        report_78[label]["recall"] for label in class_labels
    ],
    "12_Feature_Recall": [
        report_12[label]["recall"] for label in class_labels
    ],
    "78_Feature_F1": [
        report_78[label]["f1-score"] for label in class_labels
    ],
    "12_Feature_F1": [
        report_12[label]["f1-score"] for label in class_labels
    ]
})

print("=" * 70)
print("CLASS-WISE PERFORMANCE COMPARISON")
print("=" * 70)

print(
    classwise_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nClass mapping:")
print("Class 0 = BENIGN")
print("Class 1 = Bot")
print("Class 2 = DDoS")
print("Class 3 = PortScan")

CLASS-WISE PERFORMANCE COMPARISON
Class  78_Feature_Precision  12_Feature_Precision  78_Feature_Recall  12_Feature_Recall  78_Feature_F1  12_Feature_F1
    0              0.999510              0.999516           0.999644           0.999449       0.999577       0.999482
    1              0.849727              0.783920           0.795396           0.797954       0.821664       0.790875
    2              0.999688              0.999688           0.999766           0.999805       0.999727       0.999746
    3              0.999945              0.999725           0.999890           0.999835       0.999917       0.999780

Class mapping:
Class 0 = BENIGN
Class 1 = Bot
Class 2 = DDoS
Class 3 = PortScan


## 29. Compare Confusion Matrices

Confusion matrices are generated for both Decision Tree models to examine the distribution of correct and incorrect predictions across the four traffic classes.

The same class ordering is used for both representations to make the comparison directly interpretable.

In [26]:
# 29. Compare confusion matrices

from sklearn.metrics import confusion_matrix

# Define consistent class order
class_order = [0, 1, 2, 3]

# Generate confusion matrices
cm_78 = confusion_matrix(
    y_test_78_encoded,
    y_pred_78,
    labels=class_order
)

cm_12 = confusion_matrix(
    y_test_selected,
    y_pred_12,
    labels=class_order
)

# Create labeled DataFrames
cm_78_df = pd.DataFrame(
    cm_78,
    index=["Actual 0", "Actual 1", "Actual 2", "Actual 3"],
    columns=["Predicted 0", "Predicted 1", "Predicted 2", "Predicted 3"]
)

cm_12_df = pd.DataFrame(
    cm_12,
    index=["Actual 0", "Actual 1", "Actual 2", "Actual 3"],
    columns=["Predicted 0", "Predicted 1", "Predicted 2", "Predicted 3"]
)

print("=" * 70)
print("CONFUSION MATRIX — 78 ORIGINAL FEATURES")
print("=" * 70)
print(cm_78_df)

print("\n" + "=" * 70)
print("CONFUSION MATRIX — 12 SELECTED FEATURES")
print("=" * 70)
print(cm_12_df)

print("\nClass mapping:")
print("Class 0 = BENIGN")
print("Class 1 = Bot")
print("Class 2 = DDoS")
print("Class 3 = PortScan")

CONFUSION MATRIX — 78 ORIGINAL FEATURES
          Predicted 0  Predicted 1  Predicted 2  Predicted 3
Actual 0       179515           55            8            1
Actual 1           80          311            0            0
Actual 2            6            0        25597            0
Actual 3            2            0            0        18162

CONFUSION MATRIX — 12 SELECTED FEATURES
          Predicted 0  Predicted 1  Predicted 2  Predicted 3
Actual 0       179480           86            8            5
Actual 1           79          312            0            0
Actual 2            5            0        25598            0
Actual 3            3            0            0        18161

Class mapping:
Class 0 = BENIGN
Class 1 = Bot
Class 2 = DDoS
Class 3 = PortScan


## 30. Quantify Feature and Computational Reduction

The reduction in feature count and the corresponding changes in training and prediction time are calculated for the two feature representations.

These measurements quantify the computational benefit of reducing the feature space from 78 original features to 12 selected features.

In [27]:
# 30. Quantify feature and computational reduction

# Feature counts
features_78 = X_train_78_clean.shape[1]
features_12 = X_train_selected.shape[1]

# Feature reduction
feature_reduction_count = features_78 - features_12
feature_reduction_percentage = (
    feature_reduction_count / features_78
) * 100

# Training time comparison
training_time_reduction = training_time_78 - training_time_12
training_time_reduction_percentage = (
    training_time_reduction / training_time_78
) * 100

# Prediction time comparison
prediction_time_reduction = prediction_time_78 - prediction_time_12
prediction_time_reduction_percentage = (
    prediction_time_reduction / prediction_time_78
) * 100

# Speed-up factors
training_speedup = training_time_78 / training_time_12
prediction_speedup = prediction_time_78 / prediction_time_12

print("=" * 70)
print("FEATURE AND COMPUTATIONAL REDUCTION")
print("=" * 70)

print("\nFeature reduction:")
print("Original features:", features_78)
print("Selected features:", features_12)
print("Features removed:", feature_reduction_count)
print(f"Feature reduction: {feature_reduction_percentage:.2f}%")

print("\nTraining time:")
print(f"78-feature model: {training_time_78:.4f} seconds")
print(f"12-feature model: {training_time_12:.4f} seconds")
print(f"Time reduced: {training_time_reduction:.4f} seconds")
print(f"Training time reduction: {training_time_reduction_percentage:.2f}%")
print(f"Training speed-up: {training_speedup:.2f}x")

print("\nPrediction time:")
print(f"78-feature model: {prediction_time_78:.4f} seconds")
print(f"12-feature model: {prediction_time_12:.4f} seconds")
print(f"Time reduced: {prediction_time_reduction:.4f} seconds")
print(f"Prediction time reduction: {prediction_time_reduction_percentage:.2f}%")
print(f"Prediction speed-up: {prediction_speedup:.2f}x")

FEATURE AND COMPUTATIONAL REDUCTION

Feature reduction:
Original features: 78
Selected features: 12
Features removed: 66
Feature reduction: 84.62%

Training time:
78-feature model: 56.3488 seconds
12-feature model: 7.5103 seconds
Time reduced: 48.8385 seconds
Training time reduction: 86.67%
Training speed-up: 7.50x

Prediction time:
78-feature model: 0.2716 seconds
12-feature model: 0.0791 seconds
Time reduced: 0.1925 seconds
Prediction time reduction: 70.87%
Prediction speed-up: 3.43x


## 31. Build the Complete Comparison Table

This section combines the predictive performance and computational results of the 78-feature and 12-feature Decision Tree models into a single comparison table.


In [31]:
# Create a clean comparison table
comparison_results = pd.DataFrame({
    "Metric": [
        "Number of Features",
        "Training Time (seconds)",
        "Prediction Time (seconds)",
        "Accuracy (%)",
        "Weighted Precision (%)",
        "Weighted Recall (%)",
        "Weighted F1 (%)",
        "Macro F1 (%)"
    ],
    "78 Original Features": [
        X_train_78_clean.shape[1],
        training_time_78,
        prediction_time_78,
        accuracy_78 * 100,
        weighted_precision_78 * 100,
        weighted_recall_78 * 100,
        weighted_f1_78 * 100,
        macro_f1_78 * 100
    ],
    "12 Selected Features": [
        X_train_selected.shape[1],
        training_time_12,
        prediction_time_12,
        accuracy_12 * 100,
        weighted_precision_12 * 100,
        weighted_recall_12 * 100,
        weighted_f1_12 * 100,
        macro_f1_12 * 100
    ]
})

print("=" * 75)
print("COMPLETE 78-FEATURE VS 12-FEATURE COMPARISON")
print("=" * 75)

# Display without the DataFrame index
print(comparison_results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

COMPLETE 78-FEATURE VS 12-FEATURE COMPARISON
                   Metric  78 Original Features  12 Selected Features
       Number of Features               78.0000               12.0000
  Training Time (seconds)               56.3488                7.5103
Prediction Time (seconds)                0.2716                0.0791
             Accuracy (%)               99.9321               99.9169
   Weighted Precision (%)               99.9304               99.9175
      Weighted Recall (%)               99.9321               99.9169
          Weighted F1 (%)               99.9311               99.9172
             Macro F1 (%)               95.5221               94.7471


## 32. Calculate Performance Differences

This section calculates the difference in predictive performance between the 78 original features and the 12 selected features. The differences are expressed in percentage points to clearly quantify the effect of feature selection.


In [32]:
# Calculate performance differences between the two feature representations

accuracy_difference = (accuracy_12 - accuracy_78) * 100
weighted_precision_difference = (weighted_precision_12 - weighted_precision_78) * 100
weighted_recall_difference = (weighted_recall_12 - weighted_recall_78) * 100
weighted_f1_difference = (weighted_f1_12 - weighted_f1_78) * 100
macro_f1_difference = (macro_f1_12 - macro_f1_78) * 100

print("=" * 75)
print("PERFORMANCE DIFFERENCE: 12 FEATURES vs 78 FEATURES")
print("=" * 75)

print(f"Accuracy difference:              {accuracy_difference:+.4f} percentage points")
print(f"Weighted Precision difference:    {weighted_precision_difference:+.4f} percentage points")
print(f"Weighted Recall difference:       {weighted_recall_difference:+.4f} percentage points")
print(f"Weighted F1 difference:           {weighted_f1_difference:+.4f} percentage points")
print(f"Macro F1 difference:              {macro_f1_difference:+.4f} percentage points")

PERFORMANCE DIFFERENCE: 12 FEATURES vs 78 FEATURES
Accuracy difference:              -0.0152 percentage points
Weighted Precision difference:    -0.0128 percentage points
Weighted Recall difference:       -0.0152 percentage points
Weighted F1 difference:           -0.0139 percentage points
Macro F1 difference:              -0.7751 percentage points


## 33. Generate Final Experimental Findings

This section summarizes the experimental comparison between the 78 original features and the 12 selected features. The findings consider both predictive performance and computational efficiency.


In [33]:
# Calculate final experimental findings

feature_reduction_percentage = (
    (X_train_78_clean.shape[1] - X_train_selected.shape[1])
    / X_train_78_clean.shape[1]
) * 100

training_time_reduction_percentage = (
    (training_time_78 - training_time_12)
    / training_time_78
) * 100

prediction_time_reduction_percentage = (
    (prediction_time_78 - prediction_time_12)
    / prediction_time_78
) * 100

training_speedup = training_time_78 / training_time_12
prediction_speedup = prediction_time_78 / prediction_time_12

print("=" * 80)
print("FINAL EXPERIMENTAL FINDINGS")
print("=" * 80)

print("\n1. FEATURE REDUCTION")
print("-" * 80)
print(f"Original features:              {X_train_78_clean.shape[1]}")
print(f"Selected features:              {X_train_selected.shape[1]}")
print(f"Features removed:               {X_train_78_clean.shape[1] - X_train_selected.shape[1]}")
print(f"Feature reduction:              {feature_reduction_percentage:.2f}%")

print("\n2. COMPUTATIONAL EFFICIENCY")
print("-" * 80)
print(f"Training time reduction:        {training_time_reduction_percentage:.2f}%")
print(f"Training speed-up:              {training_speedup:.2f}x")
print(f"Prediction time reduction:      {prediction_time_reduction_percentage:.2f}%")
print(f"Prediction speed-up:             {prediction_speedup:.2f}x")

print("\n3. PREDICTIVE PERFORMANCE")
print("-" * 80)
print(f"Accuracy difference:             {accuracy_difference:+.4f} percentage points")
print(f"Weighted Precision difference:   {weighted_precision_difference:+.4f} percentage points")
print(f"Weighted Recall difference:      {weighted_recall_difference:+.4f} percentage points")
print(f"Weighted F1 difference:          {weighted_f1_difference:+.4f} percentage points")
print(f"Macro F1 difference:             {macro_f1_difference:+.4f} percentage points")

print("\n4. EXPERIMENTAL INTERPRETATION")
print("-" * 80)
print(
    f"The 12-feature representation reduced the feature space by "
    f"{feature_reduction_percentage:.2f}% and reduced training time by "
    f"{training_time_reduction_percentage:.2f}%, while the Macro F1 score "
    f"changed by {macro_f1_difference:+.4f} percentage points."
)

print(
    f"Prediction time was reduced by {prediction_time_reduction_percentage:.2f}%, "
    f"corresponding to a {prediction_speedup:.2f}x speed-up."
)

FINAL EXPERIMENTAL FINDINGS

1. FEATURE REDUCTION
--------------------------------------------------------------------------------
Original features:              78
Selected features:              12
Features removed:               66
Feature reduction:              84.62%

2. COMPUTATIONAL EFFICIENCY
--------------------------------------------------------------------------------
Training time reduction:        86.67%
Training speed-up:              7.50x
Prediction time reduction:      70.87%
Prediction speed-up:             3.43x

3. PREDICTIVE PERFORMANCE
--------------------------------------------------------------------------------
Accuracy difference:             -0.0152 percentage points
Weighted Precision difference:   -0.0128 percentage points
Weighted Recall difference:      -0.0152 percentage points
Weighted F1 difference:          -0.0139 percentage points
Macro F1 difference:             -0.7751 percentage points

4. EXPERIMENTAL INTERPRETATION
-------------------------

## 34. Final Validation and Save Comparison Results

The final validation confirms that both feature representations contain the same evaluation samples and target distribution, that the comparison metrics were successfully calculated, and that the experimental results are saved for future analysis and reporting.


In [34]:
# ============================================================
# FINAL VALIDATION
# ============================================================

print("=" * 80)
print("FINAL VALIDATION AND RESULT SAVING")
print("=" * 80)

# 1. Validate sample counts
sample_count_match = (
    len(X_train_78_clean) == len(X_train_selected)
    and len(X_test_78_clean) == len(X_test_selected)
)

# 2. Validate target counts
target_alignment = (
    y_train_78_encoded.value_counts().sort_index().equals(
        y_train_selected.value_counts().sort_index()
    )
    and
    y_test_78_encoded.value_counts().sort_index().equals(
        y_test_selected.value_counts().sort_index()
    )
)

# 3. Validate predictions
prediction_length_match = (
    len(y_pred_78) == len(y_test_78_encoded)
    and len(y_pred_12) == len(y_test_selected)
)

# 4. Validate feature counts
feature_count_check = (
    X_train_78_clean.shape[1] == 78
    and X_train_selected.shape[1] == 12
)

# 5. Validate missing and infinite values
data_quality_check = (
    X_train_78_clean.isna().sum().sum() == 0
    and X_test_78_clean.isna().sum().sum() == 0
    and np.isinf(X_train_78_clean.to_numpy()).sum() == 0
    and np.isinf(X_test_78_clean.to_numpy()).sum() == 0
)

# 6. Overall validation
final_validation = all([
    sample_count_match,
    target_alignment,
    prediction_length_match,
    feature_count_check,
    data_quality_check
])

print("\nValidation Checks:")
print(f"Sample count alignment:          {sample_count_match}")
print(f"Target distribution alignment:   {target_alignment}")
print(f"Prediction length validation:    {prediction_length_match}")
print(f"Feature count validation:        {feature_count_check}")
print(f"Data quality validation:          {data_quality_check}")
print(f"\nFINAL VALIDATION:                 {final_validation}")


# ============================================================
# SAVE COMPLETE COMPARISON RESULTS
# ============================================================

# Save comparison table
comparison_results.to_csv(
    "../data/processed/78_vs_12_feature_comparison.csv",
    index=False
)

# Save performance differences
performance_differences = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Weighted Precision",
        "Weighted Recall",
        "Weighted F1",
        "Macro F1"
    ],
    "Difference_Percentage_Points": [
        accuracy_difference,
        weighted_precision_difference,
        weighted_recall_difference,
        weighted_f1_difference,
        macro_f1_difference
    ]
})

performance_differences.to_csv(
    "../data/processed/78_vs_12_performance_differences.csv",
    index=False
)

# Save computational reduction results
computational_reduction = pd.DataFrame({
    "Metric": [
        "Feature Reduction (%)",
        "Training Time Reduction (%)",
        "Training Speed-up (x)",
        "Prediction Time Reduction (%)",
        "Prediction Speed-up (x)"
    ],
    "Value": [
        feature_reduction_percentage,
        training_time_reduction_percentage,
        training_speedup,
        prediction_time_reduction_percentage,
        prediction_speedup
    ]
})

computational_reduction.to_csv(
    "../data/processed/78_vs_12_computational_reduction.csv",
    index=False
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("RESULT FILES SAVED")
print("=" * 80)

print("1. 78_vs_12_feature_comparison.csv")
print("2. 78_vs_12_performance_differences.csv")
print("3. 78_vs_12_computational_reduction.csv")

print("\n" + "=" * 80)
print("NOTEBOOK 08 COMPLETED SUCCESSFULLY")
print("=" * 80)

FINAL VALIDATION AND RESULT SAVING

Validation Checks:
Sample count alignment:          True
Target distribution alignment:   True
Prediction length validation:    True
Feature count validation:        True
Data quality validation:          True

FINAL VALIDATION:                 True

RESULT FILES SAVED
1. 78_vs_12_feature_comparison.csv
2. 78_vs_12_performance_differences.csv
3. 78_vs_12_computational_reduction.csv

NOTEBOOK 08 COMPLETED SUCCESSFULLY


# 35. 🏁 Final Experimental Conclusion

## 🔥 78 Features vs 12 Selected Features

```mermaid
flowchart LR

A["78 Original Features"]
B["Feature Selection"]
C["12 Selected Features"]

A --> B --> C
```

---

## 📉 Feature Reduction

| Measure           |     Result |
| ----------------- | ---------: |
| Original Features |     **78** |
| Selected Features |     **12** |
| Features Removed  |     **66** |
| Feature Reduction | **84.62%** |

```text
78 FEATURES
     ↓
  -66 FEATURES
     ↓
12 FEATURES
```

---

## ⚡ Computational Impact

```mermaid
flowchart TD

A["78 Features"]
B["Training: 56.35 s"]
C["Prediction: 0.272 s"]

D["12 Features"]
E["Training: 7.51 s"]
F["Prediction: 0.079 s"]

A --> B
A --> C

D --> E
D --> F
```

| Measure          | 78 Features | 12 Features |           Change |
| ---------------- | ----------: | ----------: | ---------------: |
| Training Time    |   56.3488 s |    7.5103 s |     **86.67% ↓** |
| Training Speed   |          1× |   **7.50×** | **7.50× faster** |
| Prediction Time  |    0.2716 s |    0.0791 s |     **70.87% ↓** |
| Prediction Speed |          1× |   **3.43×** | **3.43× faster** |

---

## 📊 Predictive Performance

| Metric             |  78 Features |  12 Features | Difference |
| ------------------ | -----------: | -----------: | ---------: |
| Accuracy           | **99.9321%** | **99.9169%** | −0.0152 pp |
| Weighted Precision | **99.9304%** | **99.9175%** | −0.0128 pp |
| Weighted Recall    | **99.9321%** | **99.9169%** | −0.0152 pp |
| Weighted F1        | **99.9311%** | **99.9172%** | −0.0139 pp |
| Macro F1           | **95.5221%** | **94.7471%** | −0.7751 pp |

---

## 🧠 What the Experiment Shows

```mermaid
flowchart LR

A["78 Original Features"]
B["66 Features Removed"]
C["12 Selected Features"]

D["84.62% Smaller Feature Space"]
E["86.67% Less Training Time"]
F["70.87% Less Prediction Time"]

G["Only Small Predictive Performance Change"]

A --> B --> C
C --> D
C --> E
C --> F
C --> G
```

### 🔑 Key Finding

> **Large feature reduction ≠ large performance reduction**

The 12 selected features retained most of the predictive information required by the Decision Tree while substantially reducing computational cost.

---

## ⚖️ Why a Large Performance Gap Is Not Required

```text
78 FEATURES
     │
     ├── Redundant Features
     ├── Constant Features
     ├── Highly Correlated Features
     └── Less Informative Features
              │
              ↓
      FEATURE SELECTION
              │
              ↓
       12 INFORMATIVE FEATURES
              │
              ↓
   MOST PREDICTIVE INFORMATION RETAINED
```

```text
NUMBER OF FEATURES
        ≠
AMOUNT OF USEFUL INFORMATION
```

Therefore, removing **66 features** does not imply that model performance must decrease dramatically.

---

## 🚫 No Artificial Performance Gap

```mermaid
flowchart TD

A["78 vs 12 Feature Experiment"]
B["Same Dataset"]
C["Same Train/Evaluation Split"]
D["Same Target Distribution"]
E["Same Decision Tree"]
F["Same Evaluation Metrics"]
G["Measured Results"]

A --> B
A --> C
A --> D
A --> E
A --> F
B --> G
C --> G
D --> G
E --> G
F --> G
```

```text
The objective is NOT:

"Create a large performance difference"

The objective IS:

"Measure the actual effect of feature selection"
```

The measured results are therefore reported as obtained, without artificially removing additional features or modifying the comparison to create a larger performance gap.

---

## 🔄 Complete Project Pipeline

```mermaid
flowchart TD

A["CIC-IDS2017 Dataset"]
B["01 — EDA"]
C["02 — Preprocessing"]
D["03 — Feature Selection"]
E["37 → 12 Features"]
F["04 — Model Building"]
G["05 — Hyperparameter Tuning"]
H["06 — Model Comparison"]
I["07 — Final Model Evaluation"]
J["08 — 78 vs 12 Experimental Comparison"]
K["Final Results & Analysis"]

A --> B --> C --> D --> E --> F --> G --> H --> I --> J --> K
```

---

## 🏆 Final Experimental Result

```text
78 ORIGINAL FEATURES
        ↓
   84.62% REDUCTION
        ↓
12 SELECTED FEATURES
        ↓
┌─────────────────────────────┐
│ Training Time ↓ 86.67%      │
│ Training Speed ↑ 7.50×      │
│ Prediction Time ↓ 70.87%    │
│ Prediction Speed ↑ 3.43×    │
└─────────────────────────────┘
        ↓
Small Predictive Performance Change
        ↓
FEATURE SELECTION EFFECT
EXPERIMENTALLY VALIDATED
```

---

## ✅ Final Status

| Component                 | Status                       |
| ------------------------- | ---------------------------- |
| 78-feature representation | ✅ Completed                  |
| 12-feature representation | ✅ Completed                  |
| Same data split           | ✅ Verified                   |
| Target alignment          | ✅ Verified                   |
| Data quality              | ✅ Verified                   |
| Model comparison          | ✅ Completed                  |
| Performance comparison    | ✅ Completed                  |
| Computational comparison  | ✅ Completed                  |
| Final validation          | ✅ **TRUE**                   |
| Notebook 08               | ✅ **COMPLETED SUCCESSFULLY** |

> **Final conclusion:** The feature-selection pipeline achieved a substantial reduction in dimensionality and computational cost while maintaining very high predictive performance. The observed results are reported as measured, without introducing additional experiments solely to create a larger performance gap.


• 78 → 12 features: 84.62% dimensionality reduction.

• Training time reduced by 86.67% with a 7.50× speed-up.

• Prediction time reduced by 70.87% with a 3.43× speed-up.

• Accuracy changed by only −0.0152 percentage points.

• Macro F1 changed by −0.7751 percentage points.

• 12 selected features retained most of the predictive performance.

• The major benefit of feature selection was computational efficiency,
  with only a small predictive-performance trade-off.

• Therefore, based on this experiment, feature selection was
  worthwhile for reducing model complexity and computation.